In [1]:
import time

import numpy as np

from fast_search import definitive_winner
from n_game_sim import generate_hands

In [2]:
# game = Dealer(deck=full_euchre_deck ,players=4)
# game.stack_deck(stack_cards=evaluate, player=0)
# game.stack_deck(stack_cards=up_card, player=2)
# game.deal_cards()
# hands_test = np.array([game.hand0, game.hand1, game.hand2, game.hand3])
# hands_test

In [3]:
hands_test = np.array([[[  0, 140],
        [  0, 135],
        [  0,  -9],
        [ -9,   0],
        [  9,   0]],

       [[ 13,   0],
        [  0, -14],
        [  0, 100],
        [  0, 110],
        [  0, -10]],

       [[  0,  90],
        [ 10,   0],
        [ 14,   0],
        [-12,   0],
        [ 11,   0]],

       [[-11,   0],
        [-14,   0],
        [ 12,   0],
        [  0, -13],
        [-10,   0]]])

In [4]:
# quick look at what definitive_winner does
definitive_winner(dealt_hands=hands_test, starting_player=2, caller=0, verbose=True)

Starting hands:
 [[[  0 140]
  [  0 135]
  [  0  -9]
  [ -9   0]
  [  9   0]]

 [[ 13   0]
  [  0 -14]
  [  0 100]
  [  0 110]
  [  0 -10]]

 [[  0  90]
  [ 10   0]
  [ 14   0]
  [-12   0]
  [ 11   0]]

 [[-11   0]
  [-14   0]
  [ 12   0]
  [  0 -13]
  [-10   0]]]
Trick 1: [[0, 90], [-11, 0], [0, 140], [0, 100]]  (played by [2, 3, 0, 1])
Trick 1 winner: 0
Trick 2: [[0, 135], [0, 110], [-12, 0], [-10, 0]]  (played by [0, 1, 2, 3])
Trick 2 winner: 0
Trick 3: [[9, 0], [13, 0], [14, 0], [12, 0]]  (played by [0, 1, 2, 3])
Trick 3 winner: 2
Trick 4: [[11, 0], [0, -13], [0, -9], [0, -10]]  (played by [2, 3, 0, 1])
Trick 4 winner: 2
Trick 5: [[10, 0], [-14, 0], [-9, 0], [0, -14]]  (played by [2, 3, 0, 1])
Trick 5 winner: 2
Final result: [0, 0, 2, 2, 2]


2

In [5]:
evaluate = np.array(
    [  # left bower  (Jack of clubs)
        [0, 140],
        [0, 135],    
        [0, -9],
        [-9, 0], 
        [9, 0]
    ]
)
up_card = np.array(
    [  # left bower  (Jack of clubs)
        [0, 90],  
    ]
)

In [6]:
# The solver runs ~0.5 ms/hand, and generate_hands costs about the same again,
# so this is roughly 10s of work. Raise it for a tighter confidence interval.
N_GAMES = 10_000

gen_test = generate_hands(
    n_games=N_GAMES, stack=evaluate, stack_player=0,
    up_card=up_card, up_card_player=1,
)

In [7]:
scores = np.zeros(N_GAMES, dtype=np.int64)

t0 = time.perf_counter()
for i in range(N_GAMES):
    scores[i] = definitive_winner(
        dealt_hands=gen_test[i], starting_player=2, caller=0, verbose=False
    )
elapsed = time.perf_counter() - t0

mean_score = np.mean(scores)
se = np.std(scores, ddof=1) / np.sqrt(len(scores))

ci_lower = mean_score - 1.96 * se
ci_upper = mean_score + 1.96 * se

print(f"Solved {N_GAMES} hands in {elapsed:.2f}s ({1000 * elapsed / N_GAMES:.3f} ms/hand)")
print(f"Expected value: {mean_score:.3f}")
print(f"95% CI: [{ci_lower:.3f}, {ci_upper:.3f}]")

Solved 10000 hands in 5.58s (0.558 ms/hand)
Expected value: 0.351
95% CI: [0.322, 0.381]


In [9]:
# Same hand, both solvers, verbose -- to see how the play-by-play differs.
# The first tree_search call in a fresh kernel pays its ~197s JIT warmup.
import sys

# The archived solver and its two helper modules import each other by bare
# name, so put their folder on the path rather than making them a package.
if "archive/beta_approach" not in sys.path:
    sys.path.insert(0, "archive/beta_approach")

from tree_search import definitive_winner as definitive_winner_old

bar = "=" * 72

print(bar)
print("fast_search  --  exact minimax over all legal plays")
print(bar)
new_score = definitive_winner(
    dealt_hands=hands_test, starting_player=2, caller=0, verbose=True
)
print("score:", new_score)
print()

print(bar)
print("tree_search  --  mean over heuristically filtered branches")
print(bar)
old_score = definitive_winner_old(
    dealt_hands=hands_test, starting_player=2, caller=0, verbose=True
)
print("score:", old_score)
print()

print(bar)
print("fast_search =", new_score, "   tree_search =", old_score)
print(bar)

fast_search  --  exact minimax over all legal plays
Starting hands:
 [[[  0 140]
  [  0 135]
  [  0  -9]
  [ -9   0]
  [  9   0]]

 [[ 13   0]
  [  0 -14]
  [  0 100]
  [  0 110]
  [  0 -10]]

 [[  0  90]
  [ 10   0]
  [ 14   0]
  [-12   0]
  [ 11   0]]

 [[-11   0]
  [-14   0]
  [ 12   0]
  [  0 -13]
  [-10   0]]]
Trick 1: [[0, 90], [-11, 0], [0, 140], [0, 100]]  (played by [2, 3, 0, 1])
Trick 1 winner: 0
Trick 2: [[0, 135], [0, 110], [-12, 0], [-10, 0]]  (played by [0, 1, 2, 3])
Trick 2 winner: 0
Trick 3: [[9, 0], [13, 0], [14, 0], [12, 0]]  (played by [0, 1, 2, 3])
Trick 3 winner: 2
Trick 4: [[11, 0], [0, -13], [0, -9], [0, -10]]  (played by [2, 3, 0, 1])
Trick 4 winner: 2
Trick 5: [[10, 0], [-14, 0], [-9, 0], [0, -14]]  (played by [2, 3, 0, 1])
Trick 5 winner: 2
Final result: [0, 0, 2, 2, 2]
score: 2

tree_search  --  mean over heuristically filtered branches
[-0.00816327  0.03731343  1.08376963 -0.28060413  0.03731343]
[1.07407407 1.08695652 1.07741935 1.08695652]
[1.         1.16